# 01 — Data Exploration

**Malaria Cell Detection Project**  
MITS, Andhra Pradesh · B.Tech Final Year Project 2024–2025

This notebook covers:
- Dataset overview and class distribution
- Sample image visualisation (parasitized vs uninfected)
- Image dimension analysis
- Pixel intensity distributions

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Config
DATA_DIR     = '../data'
CLASSES      = ['Parasitized', 'Uninfected']
COLORS       = {'Parasitized': '#f85149', 'Uninfected': '#3fb950'}

print('Libraries loaded successfully.')

## 1. Dataset Overview

In [ ]:
counts = {}
for cls in CLASSES:
    path = os.path.join(DATA_DIR, cls)
    files = [f for f in os.listdir(path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    counts[cls] = len(files)
    print(f'{cls:15s}: {len(files):,} images')

print(f'\nTotal         : {sum(counts.values()):,} images')
print(f'Class balance : {counts["Parasitized"] / sum(counts.values()) * 100:.1f}% / {counts["Uninfected"] / sum(counts.values()) * 100:.1f}%')

In [ ]:
# Class distribution bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
bars = axes[0].bar(counts.keys(), counts.values(),
                   color=[COLORS[c] for c in counts.keys()],
                   edgecolor='white', linewidth=0.5, width=0.5)
for bar, (cls, val) in zip(bars, counts.items()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{val:,}', ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Images')
axes[0].set_ylim(0, max(counts.values()) * 1.15)
axes[0].grid(axis='y', alpha=0.3)

# Pie chart
axes[1].pie(counts.values(), labels=counts.keys(),
            colors=[COLORS[c] for c in counts.keys()],
            autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 11})
axes[1].set_title('Class Balance', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('The dataset is perfectly balanced — no class weighting required.')

## 2. Sample Images

In [ ]:
def load_samples(cls, n=8):
    path  = os.path.join(DATA_DIR, cls)
    files = sorted(os.listdir(path))[:n]
    imgs  = []
    for f in files:
        img = cv2.imread(os.path.join(path, f))
        if img is not None:
            imgs.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    return imgs

fig, axes = plt.subplots(2, 8, figsize=(18, 5))
fig.suptitle('Sample Blood Smear Images', fontsize=14, fontweight='bold', y=1.02)

for row, cls in enumerate(CLASSES):
    samples = load_samples(cls, n=8)
    axes[row, 0].set_ylabel(cls, fontsize=11, fontweight='bold',
                             color=COLORS[cls], rotation=90, labelpad=8)
    for col, img in enumerate(samples):
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        axes[row, col].set_aspect('equal')

plt.tight_layout()
plt.savefig('../results/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('Top row: Parasitized (infected) | Bottom row: Uninfected (healthy)')

## 3. Image Dimension Analysis

In [ ]:
# Sample 500 images per class to check dimensions
widths, heights = [], []
for cls in CLASSES:
    path  = os.path.join(DATA_DIR, cls)
    files = sorted(os.listdir(path))[:500]
    for f in files:
        img = cv2.imread(os.path.join(path, f))
        if img is not None:
            h, w = img.shape[:2]
            widths.append(w)
            heights.append(h)

print(f'Width  — min: {min(widths)}  max: {max(widths)}  mean: {np.mean(widths):.0f}')
print(f'Height — min: {min(heights)}  max: {max(heights)}  mean: {np.mean(heights):.0f}')
print(f'\nImages vary in size → resizing to 128×128 during preprocessing is necessary.')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths,  bins=30, color='#58a6ff', edgecolor='white', linewidth=0.5)
axes[0].set_title('Width Distribution',  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Pixels'); axes[0].set_ylabel('Count')
axes[1].hist(heights, bins=30, color='#3fb950', edgecolor='white', linewidth=0.5)
axes[1].set_title('Height Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Pixels')
for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../results/image_dimensions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Pixel Intensity Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
channels  = ['Red', 'Green', 'Blue']
ch_colors = ['#f85149', '#3fb950', '#58a6ff']

for ax_idx, cls in enumerate(CLASSES):
    path    = os.path.join(DATA_DIR, cls)
    files   = sorted(os.listdir(path))[:200]
    pixels  = {c: [] for c in channels}

    for f in files:
        img = cv2.imread(os.path.join(path, f))
        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            for i, ch in enumerate(channels):
                pixels[ch].extend(img_rgb[:, :, i].flatten().tolist())

    for ch, color in zip(channels, ch_colors):
        axes[ax_idx].hist(pixels[ch], bins=50, alpha=0.5, color=color,
                          label=ch, density=True)

    axes[ax_idx].set_title(f'Pixel Intensity — {cls}',
                            fontsize=12, fontweight='bold',
                            color=COLORS[cls])
    axes[ax_idx].set_xlabel('Pixel Value (0–255)')
    axes[ax_idx].set_ylabel('Density')
    axes[ax_idx].legend()
    axes[ax_idx].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../results/pixel_intensity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Parasitized cells show a distinct intensity shift — the dark parasite staining is visible in the Red channel.')

## Summary

| Finding | Detail |
|---|---|
| Total images | 27,558 |
| Class balance | Perfectly balanced (50% / 50%) |
| Image dimensions | Variable (~130–150px) — standardising to 128×128 |
| Colour channel | RGB; parasitized cells show distinct staining patterns |
| Preprocessing needed | Resize, normalise, augment |

Proceed to **02_preprocessing.ipynb** →